In [18]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
)
from sklearn.naive_bayes import GaussianNB

# Optional Models
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

try:
    from catboost import CatBoostClassifier
    has_catboost = True
except ImportError:
    has_catboost = False

warnings.filterwarnings('ignore')

print("=========================================")
print("1. LOADING DATA & APPLYING CRITICAL FIX")
print("=========================================")

# Load Dataset
try:
    df = pd.read_csv("Indian Liver Patient Dataset (ILPD).csv")
    print("Dataset loaded successfully.")
except FileNotFoundError:
    raise FileNotFoundError("Error: 'Indian Liver Patient Dataset (ILPD).csv' not found in the current directory.")

# ---------------------------------------------------------
# CRITICAL FIX: Unscramble CSV Headers BEFORE preprocessing
# ---------------------------------------------------------
column_correction_map = {
    'tot_proteins': 'alkphos',
    'albumin': 'sgpt',
    'ag_ratio': 'sgot',
    'sgpt': 'tot_proteins',
    'sgot': 'albumin',
    'alkphos': 'ag_ratio'
}
df = df.rename(columns=column_correction_map)
print("Headers successfully unscrambled.")

print("\n=========================================")
print("2. DATA PREPROCESSING")
print("=========================================")

# Remove duplicate rows
initial_shape = df.shape
df = df.drop_duplicates()
print(f"Removed {initial_shape[0] - df.shape[0]} duplicate rows.")

# Verify and enforce required features
features = [
    'age', 'gender', 'tot_bilirubin', 'direct_bilirubin', 
    'tot_proteins', 'albumin', 'ag_ratio', 'sgpt', 'sgot', 'alkphos'
]
target = 'is_patient'

# Keep only required columns
df = df[features + [target]].copy()

# Map Target: 1 = Liver Disease, 2 = No Liver Disease -> 1 = Liver Disease, 0 = No Liver Disease
df[target] = df[target].map({1: 1, 2: 0})
print("Target mapped: 1 -> 1 (Liver Disease), 2 -> 0 (Healthy).")

# Encode gender: Male=1, Female=0
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0, 'male': 1, 'female': 0})
print("Gender encoded: Male -> 1, Female -> 0.")

# Handle missing values 
df = df.dropna()
print(f"Dropped missing values. Final dataset shape: {df.shape}")

X = df[features]
y = df[target]

print("\n=========================================")
print("3 & 4. SPLITTING & SCALING")
print("=========================================")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Data split into train and test sets (Stratified, 80/20).")

# Scale data explicitly for models that need it
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
print("StandardScaler applied for distance/linear based models.")

print("\n=========================================")
print("5 & 6. MODEL TRAINING & HYPERPARAMETER TUNING")
print("=========================================")

# Define Models and param grids
models_scale = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000, random_state=42),
        {'C': [0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
    ),
    'SVM': (
        SVC(probability=True, random_state=42),
        {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}
    )
}

models_noscale = {
    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [None, 3, 5, 10], 'min_samples_split': [2, 5]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
    ),
    'AdaBoost': (
        AdaBoostClassifier(random_state=42),
        {'n_estimators': [50, 100], 'learning_rate': [0.1, 1.0]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=42),
        {'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
    ),
    'Gaussian Naive Bayes': (
        GaussianNB(),
        {} 
    )
}

if has_xgb:
    models_noscale['XGBoost'] = (
        XGBClassifier(eval_metric='logloss', random_state=42),
        {'n_estimators': [50, 100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
    )
if has_catboost:
    models_noscale['CatBoost'] = (
        CatBoostClassifier(verbose=0, random_state=42),
        {'iterations': [50, 100], 'depth': [3, 5], 'learning_rate': [0.05, 0.1]}
    )

all_models = []
for name, (model, grid) in models_scale.items():
    all_models.append((name, model, grid, True))
for name, (model, grid) in models_noscale.items():
    all_models.append((name, model, grid, False))

results = []
best_estimators = {}
model_scale_map = {}

for name, model, grid, needs_scale in all_models:
    try:
        X_tr = X_train_scaled if needs_scale else X_train
        X_te = X_test_scaled if needs_scale else X_test
        
        gs = GridSearchCV(estimator=model, param_grid=grid, cv=5, scoring='f1', n_jobs=-1)
        gs.fit(X_tr, y_train)
        
        best_model = gs.best_estimator_
        y_pred = best_model.predict(X_te)
        
        try:
            y_prob = best_model.predict_proba(X_te)[:, 1]
            roc_auc = roc_auc_score(y_test, y_prob)
        except:
            roc_auc = np.nan
            
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'ROC_AUC': roc_auc,
            'Requires_Scale': needs_scale,
            'Best_Params': gs.best_params_
        })
        
        best_estimators[name] = best_model
        model_scale_map[name] = needs_scale
        print(f"[{name}] tuned and evaluated successfully.")
        
    except Exception as e:
        print(f"Failed to train {name}: {e}")

print("\n=========================================")
print("7. MODEL SELECTION & EXPORT")
print("=========================================")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['F1', 'Recall'], ascending=[False, False]).reset_index(drop=True)

best_model_info = results_df.iloc[0]
best_model_name = best_model_info['Model']
final_best_model = best_estimators[best_model_name]
final_requires_scale = model_scale_map[best_model_name]

print(f"=> Auto-selected Best Model: {best_model_name} (F1: {best_model_info['F1']:.4f})")

model_file = 'liver_model.pkl'
features_file = 'liver_features.pkl'
scaler_file = 'liver_scaler.pkl'

joblib.dump(final_best_model, model_file)
joblib.dump(features, features_file)

if final_requires_scale:
    joblib.dump(scaler, scaler_file)
    print(f"Exported: {model_file}, {features_file}, {scaler_file}")
else:
    if os.path.exists(scaler_file):
        os.remove(scaler_file)
    print(f"Exported: {model_file}, {features_file}. (Scaling not required for {best_model_name})")

print("\n=========================================")
print("8. VERIFICATION: EXTREME CLINICAL CASES")
print("=========================================")

# Create textbook synthetic cases for sanity check
extreme_cases = [
    {
        'age': 65, 'gender': 1, 
        'tot_bilirubin': 18.5, 'direct_bilirubin': 9.8, 
        'tot_proteins': 4.1, 'albumin': 1.8, 'ag_ratio': 0.4, 
        'sgpt': 850, 'sgot': 920, 'alkphos': 950, 
        'Expected_Diagnosis': 'Liver Disease (1)'
    },
    {
        'age': 28, 'gender': 0, 
        'tot_bilirubin': 0.7, 'direct_bilirubin': 0.2, 
        'tot_proteins': 7.2, 'albumin': 4.5, 'ag_ratio': 1.2, 
        'sgpt': 22, 'sgot': 25, 'alkphos': 120, 
        'Expected_Diagnosis': 'Healthy (0)'
    }
]

df_extreme = pd.DataFrame(extreme_cases)
expected_extreme = df_extreme.pop('Expected_Diagnosis')
X_extreme = df_extreme[features]

if final_requires_scale:
    X_extreme_processed = pd.DataFrame(scaler.transform(X_extreme), columns=X_extreme.columns)
else:
    X_extreme_processed = X_extreme

preds_extreme = final_best_model.predict(X_extreme_processed)
probs_extreme = final_best_model.predict_proba(X_extreme_processed)

for i in range(len(extreme_cases)):
    print(f"--- Case {i+1}: Reference -> {expected_extreme.iloc[i]} ---")
    pred_label = 'Liver Disease (1)' if preds_extreme[i] == 1 else 'Healthy (0)'
    print(f"Prediction           : {pred_label}")
    print(f"Probability Array    : [Healthy: {probs_extreme[i][0]:.4f}, Disease: {probs_extreme[i][1]:.4f}]")
    print(f"Matches Expectation? : {'YES' if pred_label == expected_extreme.iloc[i] else 'NO'}\n")

print("=========================================")
print("9. VERIFICATION: REALISTIC CASES")
print("=========================================")

realistic_test_cases = [
    {
        'age': 34, 'gender': 1,
        'tot_bilirubin': 5, 'direct_bilirubin': 0.2,
        'tot_proteins': 7.1, 'albumin': 4.2, 'ag_ratio': 1.4,
        'sgpt': 24, 'sgot': 22, 'alkphos': 78,
        'Expected_Diagnosis': 'Healthy (0)'
    },
    {
        'age': 52, 'gender': 1,
        'tot_bilirubin': 3.2, 'direct_bilirubin': 1.6,
        'tot_proteins': 5.8, 'albumin': 2.6, 'ag_ratio': 0.8,
        'sgpt': 145, 'sgot': 160, 'alkphos': 310,
        'Expected_Diagnosis': 'Liver Disease (1)'
    }
]

df_real = pd.DataFrame(realistic_test_cases)
expected_real = df_real.pop('Expected_Diagnosis')
X_real = df_real[features]

if final_requires_scale:
    X_real_processed = pd.DataFrame(scaler.transform(X_real), columns=X_real.columns)
else:
    X_real_processed = X_real

preds_real = final_best_model.predict(X_real_processed)
probs_real = final_best_model.predict_proba(X_real_processed)

for i in range(len(realistic_test_cases)):
    print(f"--- Case {i+1}: Reference -> {expected_real.iloc[i]} ---")
    pred_label = 'Liver Disease (1)' if preds_real[i] == 1 else 'Healthy (0)'
    print(f"Prediction           : {pred_label}")
    print(f"Probability Array    : [Healthy: {probs_real[i][0]:.4f}, Disease: {probs_real[i][1]:.4f}]")
    print(f"Matches Expectation? : {'YES' if pred_label == expected_real.iloc[i] else 'NO'}\n")

print("Pipeline execution complete.")

1. LOADING DATA & APPLYING CRITICAL FIX
Dataset loaded successfully.
Headers successfully unscrambled.

2. DATA PREPROCESSING
Removed 13 duplicate rows.
Target mapped: 1 -> 1 (Liver Disease), 2 -> 0 (Healthy).
Gender encoded: Male -> 1, Female -> 0.
Dropped missing values. Final dataset shape: (566, 11)

3 & 4. SPLITTING & SCALING
Data split into train and test sets (Stratified, 80/20).
StandardScaler applied for distance/linear based models.

5 & 6. MODEL TRAINING & HYPERPARAMETER TUNING
[Logistic Regression] tuned and evaluated successfully.
[SVM] tuned and evaluated successfully.
[KNN] tuned and evaluated successfully.
[Decision Tree] tuned and evaluated successfully.
[Random Forest] tuned and evaluated successfully.
[AdaBoost] tuned and evaluated successfully.
[Gradient Boosting] tuned and evaluated successfully.
[Gaussian Naive Bayes] tuned and evaluated successfully.
[XGBoost] tuned and evaluated successfully.
[CatBoost] tuned and evaluated successfully.

7. MODEL SELECTION & EXP

In [12]:
# ---------------------------------------------------------
# CRITICAL FIX: Unscramble CSV Headers
# ---------------------------------------------------------
# The original CSV headers are misaligned with the standard ILPD clinical data.
# We map the wrong headers to their correct biological features.

column_correction_map = {
    'tot_proteins': 'alkphos',
    'albumin': 'sgpt',
    'ag_ratio': 'sgot',
    'sgpt': 'tot_proteins',
    'sgot': 'albumin',
    'alkphos': 'ag_ratio'
}

df = df.rename(columns=column_correction_map)

In [13]:
# %% [markdown]
# # Extreme Cases Verification
# Run this cell to test the saved model against two synthetic, absolute-certainty clinical cases.

import pandas as pd
import numpy as np
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

print("=========================================")
print("1. LOADING ARTIFACTS")
print("=========================================")

# 1. Load the exported model, features, and scaler (if it exists)
model_file = 'model5.pkl'
features_file = 'features5.pkl'
scaler_file = 'scaler5.pkl'

try:
    model = joblib.load(model_file)
    features = joblib.load(features_file)
    print(f"Loaded model successfully: {type(model).__name__}")
    
    scaler = None
    if os.path.exists(scaler_file):
        scaler = joblib.load(scaler_file)
        print("Loaded scaler successfully.")
    else:
        print("No scaler found (Model does not require scaling).")
        
except FileNotFoundError:
    raise FileNotFoundError("Error: Required .pkl files not found. Please ensure you ran the previous training pipeline cell.")

print("\n=========================================")
print("2. DEFINING DEFINITE CASES")
print("=========================================")

# 2. Create textbook synthetic cases
extreme_cases = [
    {
        'age': 65, 
        'gender': 1, # Male
        'tot_bilirubin': 18.5, # Dangerously high (Normal < 1.2)
        'direct_bilirubin': 9.8, # Dangerously high (Normal < 0.3)
        'tot_proteins': 4.1, # Low (Normal 6.0 - 8.3)
        'albumin': 1.8, # Very low (Normal 3.4 - 5.4)
        'ag_ratio': 0.4, # Very low
        'sgpt': 850, # Extremely high (Normal < 40)
        'sgot': 920, # Extremely high (Normal < 40)
        'alkphos': 950, # Extremely high (Normal 44 - 147)
        'Expected_Condition': 'Definite Liver Disease'
    },
    {
        'age': 28, 
        'gender': 0, # Female
        'tot_bilirubin': 0.7, # Perfect normal
        'direct_bilirubin': 0.2, # Perfect normal
        'tot_proteins': 7.2, # Perfect normal
        'albumin': 4.5, # Perfect normal
        'ag_ratio': 1.2, # Perfect normal
        'sgpt': 22, # Perfect normal
        'sgot': 25, # Perfect normal
        'alkphos': 120, # Perfect normal
        'Expected_Condition': 'Definite Healthy'
    }
]

df_extreme = pd.DataFrame(extreme_cases)
expected_conditions = df_extreme.pop('Expected_Condition')

# Ensure columns strictly match the exact feature order the model expects
X_extreme = df_extreme[features]

print("Synthetic definite cases generated.")

print("\n=========================================")
print("3. INFERENCE & VERIFICATION")
print("=========================================")

# 3. Scale the features if the best model required a scaler
if scaler:
    X_processed = pd.DataFrame(scaler.transform(X_extreme), columns=X_extreme.columns)
else:
    X_processed = X_extreme

# 4. Generate Predictions and Probabilities
predictions = model.predict(X_processed)
probabilities = model.predict_proba(X_processed)

# 5. Output formatted results
def get_label_string(pred_int):
    return "Liver Disease" if pred_int == 1 else "Healthy"

for i in range(len(extreme_cases)):
    print(f"--- Scenario {i+1}: {expected_conditions.iloc[i]} ---")
    print("Clinical Values:")
    for feat in features:
        print(f"  {feat.ljust(16)}: {X_extreme.iloc[i][feat]}")
        
    pred_class = predictions[i]
    pred_probs = probabilities[i]
    confidence = max(pred_probs)
    
    expected_int = 1 if 'Disease' in expected_conditions.iloc[i] else 0
    matches_logic = "YES" if pred_class == expected_int else "NO (Warning: Model logic failed on extreme case)"
    
    print(f"\nPrediction           : {get_label_string(pred_class)}")
    print(f"Probability Array    : [Healthy: {pred_probs[0]:.4f}, Disease: {pred_probs[1]:.4f}]")
    print(f"Confidence           : {confidence * 100:.2f}%")
    print(f"Matches Expectation? : {matches_logic}\n")

print("Verification complete. The model is behaving rationally at the boundaries.")

1. LOADING ARTIFACTS
Loaded model successfully: LogisticRegression
Loaded scaler successfully.

2. DEFINING DEFINITE CASES
Synthetic definite cases generated.

3. INFERENCE & VERIFICATION
--- Scenario 1: Definite Liver Disease ---
Clinical Values:
  age             : 65.0
  gender          : 1.0
  tot_bilirubin   : 18.5
  direct_bilirubin: 9.8
  tot_proteins    : 4.1
  albumin         : 1.8
  ag_ratio        : 0.4
  sgpt            : 850.0
  sgot            : 920.0
  alkphos         : 950.0

Prediction           : Healthy
Probability Array    : [Healthy: 1.0000, Disease: 0.0000]
Confidence           : 100.00%
Matches Expectation? : NO (Warning: Model logic failed on extreme case)

--- Scenario 2: Definite Healthy ---
Clinical Values:
  age             : 28.0
  gender          : 0.0
  tot_bilirubin   : 0.7
  direct_bilirubin: 0.2
  tot_proteins    : 7.2
  albumin         : 4.5
  ag_ratio        : 1.2
  sgpt            : 22.0
  sgot            : 25.0
  alkphos         : 120.0

Prediction

In [14]:
# ---------------------------------------------------------
# CRITICAL FIX: Unscramble CSV Headers
# ---------------------------------------------------------
# The original CSV headers are misaligned with the standard ILPD clinical data.
# We map the wrong headers to their correct biological features.

column_correction_map = {
    'tot_proteins': 'alkphos',
    'albumin': 'sgpt',
    'ag_ratio': 'sgot',
    'sgpt': 'tot_proteins',
    'sgot': 'albumin',
    'alkphos': 'ag_ratio'
}

df = df.rename(columns=column_correction_map)

In [16]:
import pandas as pd
import joblib

# 1. Define the realistic test cases using clinical reference ranges
realistic_test_cases = [
    {
        'age': 34, 'gender': 1,
        'tot_bilirubin': 5, 'direct_bilirubin': 0.2,
        'tot_proteins': 7.1, 'albumin': 4.2, 'ag_ratio': 1.4,
        'sgpt': 24, 'sgot': 22, 'alkphos': 78,
        'Expected_Diagnosis': 'Healthy (0)'
    },
    {
        'age': 52, 'gender': 1,
        'tot_bilirubin': 3.2, 'direct_bilirubin': 1.6,
        'tot_proteins': 5.8, 'albumin': 2.6, 'ag_ratio': 0.8,
        'sgpt': 145, 'sgot': 160, 'alkphos': 310,
        'Expected_Diagnosis': 'Liver Disease (1)'
    }
]

df_test = pd.DataFrame(realistic_test_cases)
expected = df_test.pop('Expected_Diagnosis')

# 2. Load trained pipeline artifacts
model = joblib.load('model4.pkl')
features = joblib.load('features4.pkl')
try:
    scaler = joblib.load('scaler4.pkl')
except:
    scaler = None

# Ensure feature columns align exactly with the trained sequence
X_test_cases = df_test[features]

# Apply scaling if the model requires it
if scaler:
    X_processed = pd.DataFrame(scaler.transform(X_test_cases), columns=X_test_cases.columns)
else:
    X_processed = X_test_cases

# 3. Predict and Display Results
preds = model.predict(X_processed)
probs = model.predict_proba(X_processed)

for i in range(len(realistic_test_cases)):
    print(f"Case {i+1} Reference: {expected.iloc[i]}")
    print(f"  -> Model Prediction: {'Liver Disease (1)' if preds[i] == 1 else 'Healthy (0)'}")
    print(f"  -> Probability Distribution: [Healthy: {probs[i][0]:.3f}, Disease: {probs[i][1]:.3f}]")
    print(f"  -> Prediction Match: {'YES' if (preds[i] == 1 and '1' in expected.iloc[i]) or (preds[i] == 0 and '0' in expected.iloc[i]) else 'NO'}\n")

Case 1 Reference: Healthy (0)
  -> Model Prediction: Healthy (0)
  -> Probability Distribution: [Healthy: 1.000, Disease: 0.000]
  -> Prediction Match: YES

Case 2 Reference: Liver Disease (1)
  -> Model Prediction: Healthy (0)
  -> Probability Distribution: [Healthy: 1.000, Disease: 0.000]
  -> Prediction Match: NO

